In [ ]:
# Practice Problems Day 11

### Author:

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice today's code before the weekly homework. Weekly Homework 6 assumes you have done them. Plan on about 45 minutes.

**Solutions** are in `HW-day11-solutions.ipynb` in this folder. Try every problem yourself first, then check your work. You learn a lot more from getting stuck for a few minutes than from reading my answer!

**Reading:** Geron Appendix A (Autodiff), and chapter 11, **Glorot and He Initialization** and **Better Activation Functions**. (Remember, PyTorch calls these Xavier and Kaiming.)

GOALS:

1. Do backpropagation by hand on a tiny network.
2. Write a deep network from start to finish, with Kaiming initialization.
3. Explain vanishing gradients in words.

**How much of this should you write yourself?** Problem 1 is by hand, and the check cell is given, you just run it. Problem 2 lists the steps. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious.

**Save your work.** Commit and push when you are done, so you have your code for the weekly homework and the exam:

```bash
git add .
git commit -m "Day 11 practice"
git push
```

---

In [ ]:
# RUN THIS. The California housing data from the notes, split, scaled, and turned into tensors.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

housing = pd.read_csv("data/california_housing.csv")
X = housing.drop(columns=["MedHouseVal"])
y = housing["MedHouseVal"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42)

preprocessing = Pipeline([("scale", StandardScaler())])      # the preprocessing pipeline, fit on the training rows only
X_train_scaled = preprocessing.fit_transform(X_train)
X_valid_scaled = preprocessing.transform(X_valid)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_valid_t = torch.tensor(X_valid_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_valid_t = torch.tensor(y_valid.values, dtype=torch.float32).reshape(-1, 1)
train_data = TensorDataset(X_train_t, y_train_t)               # pairs the features with the labels for the DataLoader

print("training rows:", X_train_t.shape[0], "   validation rows:", X_valid_t.shape[0])

## Problem 1: Backpropagation by hand

This is the tiny network from the notes, with different numbers: one input, one hidden neuron with a ReLU, one output neuron.

$$x = 1, \quad y = 2, \quad w_1 = 2, \quad b_1 = -1, \quad w_2 = 0.5, \quad b_2 = 1$$

**1a.** By hand, do the forward pass: find $z$, $h$, $\hat{y}$ and the loss $L = (\hat{y} - y)^2$. Then do the backward pass and find the four derivatives $\partial L / \partial w_1$, $\partial L / \partial b_1$, $\partial L / \partial w_2$ and $\partial L / \partial b_2$. Use the pictures in the notes as your guide.

**1b.** Run the cell below to check your answers with PyTorch.

**1c.** In the cell below, change $b_1$ to $-3$ and run it again. What happens to the derivatives for $w_1$ and $b_1$? Explain why, using what you know about the ReLU.

In [ ]:
# RUN THIS. Check your answers to 1a with PyTorch.
x = torch.tensor(1.0)                          # the input
y_true = torch.tensor(2.0)                     # the right answer

w1 = torch.tensor(2.0, requires_grad=True)     # requires_grad=True tells PyTorch to keep track of the derivatives
b1 = torch.tensor(-1.0, requires_grad=True)    # for 1c, change this to -3.0
w2 = torch.tensor(0.5, requires_grad=True)
b2 = torch.tensor(1.0, requires_grad=True)

# The forward pass
z = w1 * x + b1                                # the weighted sum of the hidden neuron
h = torch.relu(z)                              # the activation
y_hat = w2 * h + b2                            # the output neuron
loss = (y_hat - y_true) ** 2                   # the squared error

# The backward pass
loss.backward()

print("z:", z.item(), "  h:", h.item(), "  y_hat:", y_hat.item(), "  loss:", loss.item())
print("dL/dw1 =", w1.grad.item(), "  dL/db1 =", b1.grad.item(), "  dL/dw2 =", w2.grad.item(), "  dL/db2 =", b2.grad.item())

*Your answers here.*

---

## Problem 2: A deep network from start to finish (steps outlined)

**What to copy, what to write:** Use the "What this looks like in real life" cell at the end of the notes as your guide, but type it out yourself. This is the pattern you need for the weekly homework.

**2a.** Write one cell that builds a network with **4 hidden layers** of 40 neurons each, with ReLU after each one, gives it Kaiming initialization, and trains it with SGD, a learning rate of 0.001, for 20 epochs. Print the validation RMSE after the first epoch and then every 5 epochs.

**2b.** Copy your cell into the next cell and delete the Kaiming initialization loop, so the network uses PyTorch's default starting weights. Run it.

**2c.** Compare the validation RMSE after 1 epoch and after 20 epochs. Which starting weights got going faster? Use what you learned in the notes to explain why.

In [ ]:
# Step 1: torch.manual_seed(42), then build the model with nn.Sequential:
#         Linear(8, 40), ReLU, then three more Linear(40, 40) each followed by ReLU, then Linear(40, 1)

# Step 2: loop over the layers. For every nn.Linear layer use nn.init.kaiming_normal_ on the weights
#         (with nonlinearity="relu") and nn.init.zeros_ on the bias

# Step 3: make the DataLoader (batch_size=32, shuffle=True), the loss function nn.MSELoss(),
#         and the optimizer torch.optim.SGD with lr=0.001

# Step 4: the training loop from the notes, 20 epochs

# Step 5: inside the epoch loop, after the batches: if epoch == 0 or (epoch + 1) % 5 == 0,
#         print the validation RMSE


In [ ]:
# 2b: your copy without the Kaiming initialization loop


*Your answers here.*

---

## Problem 3: In words

**3a.** Explain **vanishing gradients** to a friend in DATA 101 who knows what a derivative is but has never seen a neural network. Three or four sentences, no code.

**3b.** A classmate builds a network with 20 hidden layers and sigmoid activations to predict house values. After 30 epochs the validation RMSE is exactly what you get by always predicting the average. Name two things they should change, and say why each one helps.

*Your answers here.*

---

## Optional exploration

Only if you are curious. Nothing here is needed for the weekly homework.

**A.** In your 2a cell, change every `nn.ReLU()` to `nn.LeakyReLU()`. Is it any different?

**B.** In your 2a cell, replace the Kaiming line with `nn.init.normal_(layer.weight, std=1.0)`, starting weights that are much too big. What happens, and why?

In [ ]:
# your code here

*Your answers here.*